# Bottle Cap Detection — Lightweight ML Pipeline
**YOLOv8-nano trained from auto-annotations**

This notebook takes you from raw images to a deployable YOLOv8-nano detector in four stages:

| Stage | What happens |
|---|---|
| 1. Auto-annotate | Classical pipeline generates YOLO bounding boxes — no manual labelling |
| 2. Dataset | Split into train/val, write YOLO-format labels and `data.yaml` |
| 3. Train | Fine-tune YOLOv8-nano (~3MB) for ~50 epochs |
| 4. Evaluate | Visualise predictions, check mAP, export to ONNX for Jetson |

### Why YOLOv8-nano?
- ~3MB weights, ~5ms inference on CPU
- Handles varied backgrounds, angles, lighting out of the box
- Replaces the classical blob detector as Stage 1 of the DLI pipeline
- Exports to TensorRT for Jetson deployment

### Prerequisites
```
# Add to pyproject.toml dependencies
ultralytics
opencv-python-headless
numpy
matplotlib
jupyter
ipykernel
pyyaml
torch          # CPU-only is fine for nano
torchvision
```

## 1. Imports & configuration

In [ ]:
import cv2
import numpy as np
import os
import shutil
import yaml
import random
import time
from pathlib import Path
from typing import Optional, Tuple, List
import matplotlib.pyplot as plt
import matplotlib.patches as patches

print(f"OpenCV : {cv2.__version__}")

try:
    from ultralytics import YOLO
    import torch
    print(f"PyTorch: {torch.__version__}")
    print(f"Device : {'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'}")
except ImportError:
    print("ultralytics not installed — run: uv add ultralytics torch torchvision")

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
IMAGE_DIR   = Path("images")       # raw input images
DATASET_DIR = Path("dataset")      # YOLO dataset (auto-created)
RUN_DIR     = Path("runs")         # training output

# ── Dataset split ──────────────────────────────────────────────────────────
VAL_FRAC    = 0.15    # fraction of images for validation
RANDOM_SEED = 42

# ── Auto-annotation (classical pipeline) ───────────────────────────────────
DETECT_SIZE  = 500    # downsample resolution for blob detection
MIN_CIRC     = 0.45   # minimum blob circularity
MIN_AREA_FRAC = 0.03
MAX_AREA_FRAC = 0.55
HOUGH_P2_VALUES = [35, 28, 22]

# ── Training ───────────────────────────────────────────────────────────────
MODEL_SIZE  = "yolov8n.pt"   # nano — swap to yolov8s.pt for more accuracy
EPOCHS      = 50
IMG_SIZE    = 640            # YOLO input resolution
BATCH_SIZE  = 8              # reduce to 4 if memory is tight
PATIENCE    = 15             # early stopping patience

# ── Inference ──────────────────────────────────────────────────────────────
CONF_THRESH = 0.40           # minimum confidence to report a detection
IOU_THRESH  = 0.45           # NMS IoU threshold

print("Configuration loaded.")

## 2. Classical pipeline — auto-annotator

Runs the existing cap detector on every raw image and converts the detected circle to a YOLO bounding box. No manual labelling needed.

In [ ]:
# ── Classical detector (from cap_detection_pipeline.ipynb) ─────────────────

def normalise_fast(img: np.ndarray) -> np.ndarray:
    small = cv2.resize(img, (DETECT_SIZE, DETECT_SIZE), interpolation=cv2.INTER_AREA)
    hsv   = cv2.cvtColor(small, cv2.COLOR_BGR2HSV).astype(np.uint8)
    if hsv[..., 2].mean() < 100:
        t = np.array([(i / 255.0) ** 0.4 * 255 for i in range(256)], np.uint8)
        hsv[..., 2] = cv2.LUT(hsv[..., 2], t)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    hsv[..., 2] = clahe.apply(hsv[..., 2])
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)


def blob_detect_fast(norm_small, img_area_small):
    h, w = norm_small.shape[:2]
    cx_img, cy_img = w / 2, h / 2
    hsv = cv2.cvtColor(norm_small, cv2.COLOR_BGR2HSV)
    sat, val = hsv[..., 1], hsv[..., 2]
    otsu_val, _ = cv2.threshold(sat, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    thresholds   = sorted(set([max(30, int(otsu_val)), 55, 65, 75, 85]))
    k1 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (6, 6))
    k2 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
    for s_thresh in thresholds:
        mask = ((sat < s_thresh) & (val > 80)).astype(np.uint8) * 255
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k1)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  k2)
        ctrs, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        best_score, best = 0, None
        for c in ctrs:
            area = cv2.contourArea(c)
            if not (MIN_AREA_FRAC * img_area_small < area < MAX_AREA_FRAC * img_area_small):
                continue
            if len(c) < 5:
                continue
            ell = cv2.fitEllipse(c)
            (ecx, ecy), (MA, ma), _ = ell
            circ = min(MA, ma) / max(MA, ma)
            if circ < MIN_CIRC:
                continue
            dist  = np.hypot((ecx - cx_img) / w, (ecy - cy_img) / h)
            score = circ * max(0, 1 - 2 * dist) * np.log(area)
            if score > best_score:
                best_score, best = score, ell
        if best:
            return best
    return None


def find_cap(img: np.ndarray) -> Optional[dict]:
    """
    Returns dict with cx, cy, r (all in full-image pixels) or None.
    """
    h, w  = img.shape[:2]
    scale = h / DETECT_SIZE
    norm_s = normalise_fast(img)
    hs, ws = norm_s.shape[:2]
    area_s = hs * ws

    ell_s = blob_detect_fast(norm_s, area_s)
    if ell_s is None:
        raw_s = cv2.resize(img, (DETECT_SIZE, DETECT_SIZE), interpolation=cv2.INTER_AREA)
        ell_s = blob_detect_fast(raw_s, area_s)
    if ell_s is None:
        return None

    (ecx, ecy), (MA, ma), angle = ell_s
    ecx_f, ecy_f = ecx * scale, ecy * scale
    MA_f,  ma_f  = MA * scale,  ma * scale
    circ = min(MA_f, ma_f) / max(MA_f, ma_f)

    if circ < 0.88:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        win  = int(max(MA_f, ma_f) * 0.75)
        x1 = max(int(ecx_f) - win, 0);  y1 = max(int(ecy_f) - win, 0)
        x2 = min(int(ecx_f) + win, w);  y2 = min(int(ecy_f) + win, h)
        roi = cv2.GaussianBlur(gray[y1:y2, x1:x2], (13, 13), 0)
        min_r = int(min(MA_f, ma_f) / 2 * 0.7)
        max_r = int(max(MA_f, ma_f) / 2 * 1.3)
        for p2 in HOUGH_P2_VALUES:
            circles = cv2.HoughCircles(
                roi, cv2.HOUGH_GRADIENT, dp=1.2,
                minDist=200, param1=70, param2=p2,
                minRadius=min_r, maxRadius=max_r
            )
            if circles is not None:
                cx, cy, r = np.round(circles[0][0]).astype(int)
                return dict(cx=float(cx + x1), cy=float(cy + y1), r=float(r))

    r = max(MA_f, ma_f) / 2
    return dict(cx=ecx_f, cy=ecy_f, r=r)


print("Classical detector loaded.")

In [ ]:
def circle_to_yolo(cx: float, cy: float, r: float,
                   img_w: int, img_h: int,
                   padding: float = 0.08) -> Tuple[float, float, float, float]:
    """
    Convert a circle (cx, cy, r) to YOLO bbox format:
    (x_centre, y_centre, width, height) — all normalised 0–1.

    Adds padding around the circle so the bounding box includes the rim.
    """
    r_pad = r * (1 + padding)
    x1 = max(cx - r_pad, 0)
    y1 = max(cy - r_pad, 0)
    x2 = min(cx + r_pad, img_w)
    y2 = min(cy + r_pad, img_h)

    x_c = ((x1 + x2) / 2) / img_w
    y_c = ((y1 + y2) / 2) / img_h
    bw  = (x2 - x1) / img_w
    bh  = (y2 - y1) / img_h

    return x_c, y_c, bw, bh


print("YOLO conversion ready.")

In [ ]:
def auto_annotate(image_dir: Path) -> Tuple[List[Path], List[Path]]:
    """
    Run classical detector on all images in image_dir.
    Saves a .txt label file alongside each image (YOLO format, class 0).

    Returns (annotated_paths, failed_paths).
    """
    images   = sorted(image_dir.glob("*.jpg")) + sorted(image_dir.glob("*.png"))
    annotated, failed = [], []

    for path in images:
        img = cv2.imread(str(path))
        if img is None:
            failed.append(path)
            continue

        h, w = img.shape[:2]
        cap  = find_cap(img)

        if cap is None:
            print(f"  SKIP  {path.name} — no cap detected")
            failed.append(path)
            continue

        x_c, y_c, bw, bh = circle_to_yolo(cap['cx'], cap['cy'], cap['r'], w, h)

        # Write YOLO label: <class> <x_c> <y_c> <w> <h>
        label_path = path.with_suffix('.txt')
        with open(label_path, 'w') as f:
            f.write(f"0 {x_c:.6f} {y_c:.6f} {bw:.6f} {bh:.6f}\n")

        annotated.append(path)

    print(f"\nAnnotated: {len(annotated)}/{len(images)}")
    if failed:
        print(f"Failed   : {[p.name for p in failed]}")
    return annotated, failed


# ── Run auto-annotation ────────────────────────────────────────────────────
annotated_paths, failed_paths = auto_annotate(IMAGE_DIR)

In [ ]:
def verify_annotations(image_dir: Path, n_samples: int = 6, cols: int = 3):
    """
    Spot-check auto-annotations by drawing the bounding boxes on
    a random sample of images.
    """
    label_files = sorted(image_dir.glob("*.txt"))
    samples     = random.sample(label_files, min(n_samples, len(label_files)))
    rows        = (len(samples) + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
    axes = np.array(axes).flatten()

    for ax, label_path in zip(axes, samples):
        img_path = label_path.with_suffix('.jpg')
        if not img_path.exists():
            img_path = label_path.with_suffix('.png')
        img = cv2.imread(str(img_path))
        h, w = img.shape[:2]

        with open(label_path) as f:
            parts = list(map(float, f.read().split()))
        _, x_c, y_c, bw, bh = parts

        # Convert normalised → pixel coords
        x1 = int((x_c - bw / 2) * w)
        y1 = int((y_c - bh / 2) * h)
        x2 = int((x_c + bw / 2) * w)
        y2 = int((y_c + bh / 2) * h)

        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(rgb)
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                  linewidth=2, edgecolor='#00DCF0', facecolor='none')
        ax.add_patch(rect)
        ax.set_title(img_path.name, fontsize=8)
        ax.axis('off')

    for ax in axes[len(samples):]:
        ax.axis('off')

    plt.suptitle('Auto-annotation spot check', fontsize=11)
    plt.tight_layout()
    plt.show()
    plt.close('all')


random.seed(RANDOM_SEED)
verify_annotations(IMAGE_DIR)

## 3. Build YOLO dataset

Splits annotated images into `train/` and `val/` and writes the `data.yaml` that YOLO needs.

In [ ]:
def build_dataset(
    annotated_paths: List[Path],
    dataset_dir: Path,
    val_frac: float = 0.15,
    seed: int = 42
) -> Path:
    """
    Creates the YOLO dataset directory structure:

        dataset/
            images/
                train/   ← training images
                val/     ← validation images
            labels/
                train/   ← training labels (.txt)
                val/     ← validation labels (.txt)
            data.yaml

    Returns path to data.yaml.
    """
    # Clean and recreate
    if dataset_dir.exists():
        shutil.rmtree(dataset_dir)

    for split in ['train', 'val']:
        (dataset_dir / 'images' / split).mkdir(parents=True)
        (dataset_dir / 'labels' / split).mkdir(parents=True)

    # Shuffle and split
    paths = annotated_paths.copy()
    random.seed(seed)
    random.shuffle(paths)
    n_val  = max(1, int(len(paths) * val_frac))
    splits = {'val': paths[:n_val], 'train': paths[n_val:]}

    for split, split_paths in splits.items():
        for img_path in split_paths:
            # Copy image
            shutil.copy(img_path,
                        dataset_dir / 'images' / split / img_path.name)
            # Copy label
            label_src = img_path.with_suffix('.txt')
            if label_src.exists():
                shutil.copy(label_src,
                            dataset_dir / 'labels' / split / label_src.name)

    # Write data.yaml
    yaml_path = dataset_dir / 'data.yaml'
    data_yaml = {
        'path'  : str(dataset_dir.resolve()),
        'train' : 'images/train',
        'val'   : 'images/val',
        'nc'    : 1,
        'names' : ['bottle_cap'],
    }
    with open(yaml_path, 'w') as f:
        yaml.dump(data_yaml, f, default_flow_style=False, sort_keys=False)

    n_train = len(splits['train'])
    n_val_  = len(splits['val'])
    print(f"Dataset built → {dataset_dir}")
    print(f"  Train : {n_train} images")
    print(f"  Val   : {n_val_} images")
    print(f"  YAML  : {yaml_path}")
    return yaml_path


yaml_path = build_dataset(annotated_paths, DATASET_DIR, VAL_FRAC, RANDOM_SEED)

## 4. Train YOLOv8-nano

In [ ]:
# ── Determine best available device ───────────────────────────────────────
import torch

if torch.cuda.is_available():
    device = '0'        # CUDA GPU
elif torch.backends.mps.is_available():
    device = 'mps'      # Apple Silicon
else:
    device = 'cpu'

print(f"Training device: {device}")

# Note: MPS is supported but occasionally slower than CPU for small models
# If training is slow on MPS, set device = 'cpu'

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────
model = YOLO(MODEL_SIZE)   # downloads pretrained nano weights on first run

results = model.train(
    data       = str(yaml_path),
    epochs     = EPOCHS,
    imgsz      = IMG_SIZE,
    batch      = BATCH_SIZE,
    device     = device,
    patience   = PATIENCE,    # early stopping
    project    = str(RUN_DIR),
    name       = 'cap_detector',
    exist_ok   = True,

    # Augmentations — tuned for cap images
    hsv_h      = 0.01,   # minimal hue shift (caps are always white/translucent)
    hsv_s      = 0.4,    # saturation variance
    hsv_v      = 0.4,    # brightness variance — important for lighting variation
    degrees    = 180,    # full rotation (caps can be any orientation)
    translate  = 0.1,
    scale      = 0.3,
    fliplr     = 0.5,
    flipud     = 0.5,
    mosaic     = 0.5,    # mosaic augmentation

    # Output
    save       = True,
    plots      = True,
    verbose    = True,
)

best_weights = Path(results.save_dir) / 'weights' / 'best.pt'
print(f"\nBest weights: {best_weights}")

## 5. Evaluate

In [ ]:
# ── Validation metrics ─────────────────────────────────────────────────────
best_model = YOLO(best_weights)
metrics    = best_model.val(data=str(yaml_path), imgsz=IMG_SIZE, device=device)

print(f"\nmAP50      : {metrics.box.map50:.3f}")
print(f"mAP50-95   : {metrics.box.map:.3f}")
print(f"Precision  : {metrics.box.mp:.3f}")
print(f"Recall     : {metrics.box.mr:.3f}")

In [ ]:
# ── Training curves ────────────────────────────────────────────────────────
results_csv = Path(results.save_dir) / 'results.csv'

if results_csv.exists():
    import csv
    with open(results_csv) as f:
        reader  = csv.DictReader(f)
        rows    = list(reader)
        cols    = {k.strip(): [float(r[k]) for r in rows] for k in rows[0]}

    epochs_x = cols['epoch']

    fig, axes = plt.subplots(1, 3, figsize=(13, 3))

    axes[0].plot(epochs_x, cols['train/box_loss'], label='train')
    axes[0].plot(epochs_x, cols['val/box_loss'],   label='val')
    axes[0].set_title('Box loss'); axes[0].legend()

    axes[1].plot(epochs_x, cols['metrics/precision(B)'], label='precision')
    axes[1].plot(epochs_x, cols['metrics/recall(B)'],    label='recall')
    axes[1].set_title('Precision / Recall'); axes[1].legend()

    axes[2].plot(epochs_x, cols['metrics/mAP50(B)'],    label='mAP50')
    axes[2].plot(epochs_x, cols['metrics/mAP50-95(B)'], label='mAP50-95')
    axes[2].set_title('mAP'); axes[2].legend()

    for ax in axes:
        ax.set_xlabel('Epoch')
        ax.grid(alpha=0.3)

    plt.suptitle('Training curves', fontsize=11)
    plt.tight_layout()
    plt.show()
    plt.close('all')

In [ ]:
# ── Visual predictions on validation set ──────────────────────────────────
def show_predictions(
    model,
    image_paths: List[Path],
    n_samples: int = 8,
    cols: int = 4,
    conf: float = CONF_THRESH
):
    samples = random.sample(image_paths, min(n_samples, len(image_paths)))
    rows    = (len(samples) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.5, rows * 3.5))
    axes = np.array(axes).flatten()

    for ax, path in zip(axes, samples):
        img = cv2.imread(str(path))
        h, w = img.shape[:2]

        preds = model.predict(img, conf=conf, iou=IOU_THRESH,
                              imgsz=IMG_SIZE, verbose=False)[0]

        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(rgb)

        for box in preds.boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            conf_val = box.conf[0].item()
            rect = patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=2, edgecolor='#00DCF0', facecolor='none'
            )
            ax.add_patch(rect)
            ax.text(x1, y1 - 5, f'{conf_val:.2f}',
                    color='#00DCF0', fontsize=8, fontweight='bold')

        detected = len(preds.boxes) > 0
        ax.set_title(f"{path.name}\n{'✓' if detected else '✗ missed'}",
                     fontsize=7,
                     color='green' if detected else 'red')
        ax.axis('off')

    for ax in axes[len(samples):]:
        ax.axis('off')

    plt.suptitle('Predictions on validation images', fontsize=11)
    plt.tight_layout()
    plt.show()
    plt.close('all')


val_images = sorted((DATASET_DIR / 'images' / 'val').glob('*.jpg'))
show_predictions(best_model, val_images)

## 6. Benchmark inference speed

In [ ]:
def benchmark_inference(
    model,
    image_paths: List[Path],
    n_runs: int = 20,
    n_warmup: int = 5
):
    """
    Time YOLO inference across all images.
    Reports mean, p50, p95 latency in ms and fps.
    """
    times = []

    for path in image_paths[:n_warmup]:
        img = cv2.imread(str(path))
        model.predict(img, imgsz=IMG_SIZE, verbose=False)

    for path in image_paths:
        img = cv2.imread(str(path))
        for _ in range(n_runs // len(image_paths) + 1):
            t0 = time.perf_counter()
            model.predict(img, conf=CONF_THRESH, imgsz=IMG_SIZE, verbose=False)
            times.append((time.perf_counter() - t0) * 1000)

    times = np.array(times)
    print(f"Inference timing ({device}, imgsz={IMG_SIZE}):")
    print(f"  Mean  : {times.mean():.1f}ms")
    print(f"  p50   : {np.percentile(times, 50):.1f}ms")
    print(f"  p95   : {np.percentile(times, 95):.1f}ms")
    print(f"  fps   : {1000 / times.mean():.1f}")


all_images = sorted(IMAGE_DIR.glob('*.jpg'))
benchmark_inference(best_model, all_images)

## 7. Export for deployment

In [ ]:
# ── Export to ONNX ────────────────────────────────────────────────────────
# ONNX runs on any hardware — CPU, GPU, Jetson (via TensorRT backend)
onnx_path = best_model.export(format='onnx', imgsz=IMG_SIZE, simplify=True)
print(f"ONNX model: {onnx_path}")

In [ ]:
# ── Jetson: export to TensorRT ────────────────────────────────────────────
# Run this on the Jetson itself, not on your Mac.
# TensorRT optimises specifically for the target GPU.
#
# On the Jetson:
#   pip install ultralytics
#   yolo export model=best.pt format=engine imgsz=640 device=0
#
# Then load with:
#   model = YOLO('best.engine')

print("TensorRT export: run on Jetson — see comment above.")

## 8. Drop-in replacement for the classical detector

Once trained, swap the YOLO detector into the DLI pipeline as Stage 1.

In [ ]:
def detect_cap_yolo(
    img: np.ndarray,
    model,
    conf: float = CONF_THRESH
) -> Optional[dict]:
    """
    Drop-in replacement for find_cap() using the trained YOLO model.

    Returns dict with cx, cy, r (full-image pixels) or None.
    Same interface as the classical detector — rest of pipeline unchanged.
    """
    preds = model.predict(img, conf=conf, iou=IOU_THRESH,
                          imgsz=IMG_SIZE, verbose=False)[0]

    if len(preds.boxes) == 0:
        return None

    # Take highest-confidence detection
    best_box = preds.boxes[preds.boxes.conf.argmax()]
    x1, y1, x2, y2 = best_box.xyxy[0].tolist()

    cx = (x1 + x2) / 2
    cy = (y1 + y2) / 2
    r  = min(x2 - x1, y2 - y1) / 2   # inscribed circle radius

    return dict(cx=cx, cy=cy, r=r, conf=float(best_box.conf[0]))


# ── Example: full DLI pipeline with YOLO Stage 1 ──────────────────────────
def process_image_ml(
    img: np.ndarray,
    yolo_model,
    crop_size: int = 256,
    padding: float = 0.08,
    bg: tuple = (180, 180, 180)
) -> dict:
    """
    Full pipeline using YOLO for cap detection.
    Rim radius measurement and masked crop are unchanged.
    """
    result = dict(status='no_cap', crop=None, mask=None,
                  centre=None, r_true=None, conf=None)

    cap = detect_cap_yolo(img, yolo_model)
    if cap is None:
        return result

    cx, cy, r_approx = cap['cx'], cap['cy'], int(cap['r'])

    # Rim measurement (unchanged from classical pipeline)
    r_true = find_rim_radius_fast(img, cx, cy, r_approx)

    # Masked crop (unchanged)
    crop, mask = make_masked_crop(img, cx, cy, r_true, crop_size, padding, bg)

    result.update(
        status = 'ok',
        crop   = crop,
        mask   = mask,
        centre = (int(cx), int(cy)),
        r_true = r_true,
        conf   = cap['conf'],
    )
    return result


print("YOLO-based pipeline ready.")
print("Call: result = process_image_ml(img, best_model)")

In [ ]:
# ── Rim + crop helpers (copied from cap_detection_pipeline.ipynb) ─────────
# These are unchanged regardless of whether Stage 1 is classical or YOLO

RIM_SAT_MAX  = 85
RIM_VAL_MIN  = 95
N_RIM_ANGLES = 36
CROP_SIZE    = 256
PADDING      = 0.08
BG_COLOUR    = (180, 180, 180)

def find_rim_radius_fast(img, cx, cy, r_approx):
    small = cv2.resize(img, (DETECT_SIZE, DETECT_SIZE), interpolation=cv2.INTER_AREA)
    h0, w0 = img.shape[:2]
    hs, ws = small.shape[:2]
    sx = ws / w0
    cx_s, cy_s = cx * sx, cy * sx
    r_s = r_approx * sx
    norm_s = normalise_fast(img)
    hsv = cv2.cvtColor(norm_s, cv2.COLOR_BGR2HSV)
    sat, val = hsv[..., 1], hsv[..., 2]
    r_min = int(r_s * 0.60); r_max = int(r_s * 1.65)
    outer = []
    for a_deg in np.linspace(0, 360, N_RIM_ANGLES, endpoint=False):
        a = np.deg2rad(a_deg); last_cap = None
        for rr in range(r_min, r_max, 2):
            px = int(cx_s + rr * np.cos(a)); py = int(cy_s + rr * np.sin(a))
            if not (0 <= px < ws and 0 <= py < hs): break
            if sat[py, px] < RIM_SAT_MAX and val[py, px] > RIM_VAL_MIN: last_cap = rr
            elif last_cap is not None: break
        if last_cap is not None: outer.append(last_cap / sx)
    if not outer: return r_approx
    clean = [v for v in outer if v >= r_approx * 0.85]
    if len(clean) < 6: return int(np.percentile(outer, 90))
    return int(np.median(clean))


def make_masked_crop(img, cx, cy, r_true,
                     crop_size=CROP_SIZE, padding=PADDING, bg=BG_COLOUR):
    h, w  = img.shape[:2]
    r_pad = int(r_true * (1 + padding))
    x1 = max(int(cx) - r_pad, 0); y1 = max(int(cy) - r_pad, 0)
    x2 = min(int(cx) + r_pad, w); y2 = min(int(cy) + r_pad, h)
    ch, cw = y2 - y1, x2 - x1; side = max(ch, cw)
    sq = np.full((side, side, 3), bg, np.uint8)
    ox = (side - cw) // 2; oy = (side - ch) // 2
    sq[oy:oy + ch, ox:ox + cw] = img[y1:y2, x1:x2]
    crop_n = cv2.resize(sq, (crop_size, crop_size), interpolation=cv2.INTER_AREA)
    mq = np.zeros((crop_size, crop_size), np.uint8)
    scale_m = crop_size / side
    cx_m = int((int(cx) - x1 + ox) * scale_m)
    cy_m = int((int(cy) - y1 + oy) * scale_m)
    r_m  = int(r_true * scale_m)
    cv2.circle(mq, (cx_m, cy_m), r_m, 255, -1)
    crop_n[mq == 0] = bg
    return crop_n, mq


print("Rim + crop helpers ready.")

## 9. Quick-start

Uncomment and run any of the cells below once training is complete.

In [ ]:
# ── Single image inference ─────────────────────────────────────────────────
# img    = cv2.imread('images/IMG_4097.jpg')
# result = process_image_ml(img, best_model)
# print(result['status'], result['conf'])
#
# if result['status'] == 'ok':
#     plt.imshow(cv2.cvtColor(result['crop'], cv2.COLOR_BGR2RGB))
#     plt.axis('off'); plt.show(); plt.close('all')

In [ ]:
# ── Load trained model later ───────────────────────────────────────────────
# best_model = YOLO('runs/cap_detector/weights/best.pt')
# result = process_image_ml(cv2.imread('images/IMG_4097.jpg'), best_model)

In [ ]:
# ── Predict on a new folder of images ─────────────────────────────────────
# new_images = sorted(Path('new_images').glob('*.jpg'))
# for path in new_images:
#     img    = cv2.imread(str(path))
#     result = process_image_ml(img, best_model)
#     print(f"{path.name}: {result['status']}  conf={result.get('conf', '-')}")